# Independent BiLSTM Experiment — Soil Moisture Forecasting (TxSON33 dataset)

Goal: build a small, understandable **bidirectional LSTM** forecasting pipeline end to end (preprocessing →
feature engineering → normalization → windowing → BiLSTM → training → evaluation) on the TxSON33 modeling
dataset, and compare it against simple naive baselines (last value, moving average). This notebook replaces
the earlier 1D CNN version (`independent_cnn_experiment_txson33.ipynb`); everything CNN-specific
(convolutions, kernel size, pooling) has been removed.

Deliberately kept simple: one plain BiLSTM, **no attention**. The only engineered inputs are log rainfall
(current hour plus rolling 3h / 24h totals), wind vectors and calendar encodings, and forward feature
selection decides which of them are used.

## Plan summary

- **Dataset:** `TxSON33_modeling_complete.parquet` (in this folder) — 33 stations, hourly, 2014-09 through
  2026-02, with every measurement carrying a matching `*_Origin` column (`observed` / `pipeline_imputed` /
  `modeling_completion`). The dataset's own README is explicit that `modeling_completion` values are synthetic
  and "should not normally be used as validation or test ground truth" — so station/year selection below is
  driven by `*_Origin` composition, not just raw row counts.
- **Station / year:** `data_quality_summary.csv` was checked across all 33 stations for `SWC_20` and `Ppt`
  observed-origin percentage. Several stations hit 100% observed for `SWC_20`, but most either cover too short a
  span or don't line up with a full calendar year. `FD11` has the longest coverage of the 100%-`SWC_20`-observed
  stations (2014-11 to 2025-07). This version uses the same station over the three calendar
  years **2018–2020** (`START_DATE = "2018-01-01"`, `END_DATE_EXCLUSIVE = "2021-01-01"`), verified directly
  against the parquet: exactly 26,304 hourly rows (no gaps, no duplicate timestamps, no missing values);
  `SWC_20` is `observed` for all but 2 `pipeline_imputed` hours and `Ppt` for all but 1, with no
  `modeling_completion` values in either. (The earlier single-year version used 2018 only: 8,760 rows.)
- **Target:** `SWC_20` (soil water content at 20cm depth).
- **Input features:** hyperparameters are first tuned with the mandatory `SWC_20` + `Ppt_log`; greedy forward
  feature selection (Section 11) then adds candidate features (or feature groups) to those two only while
  they lower the CV RMSE, and the grid is re-run on the selected set.
- **Lookback window:** the last **24 hours** of every input feature.
- **Prediction:** a **single value** — `SWC_20` at `h` hours after the window's last hour — for each horizon
  `h` in **48h (Experiment A)** and **72h (Experiment B)**. Each horizon is tuned, feature-selected, retuned
  and evaluated independently, with its own model.
- **Offset semantics:** input window covers hours `[t, t+23]`; the target for horizon `h` is the value at
  `t + 23 + h`. Verified with real timestamps further down.
- **Naive baselines:** *last value* (persistence) and *24h moving average*, evaluated on exactly the same test
  targets as the BiLSTM. A learned model is only useful if it beats these.
- **Split:** chronological **70% train / 10% early-stopping validation / 20% final test**. The test set is
  used exactly once, for the final comparison.
- **Hyperparameter tuning:** `TimeSeriesSplit` cross-validation with `N_FOLDS` folds (2, 3 or 4; default 2)
  **inside the 70% training data only**. A grid search (`ParameterGrid`) over `num_layers` 1/2 × `hidden_size`
  32/48 × `lr` 0.001/0.005 (`batch_size` 16) — 8 combinations, up to 20 epochs with patience 5 — is run per
  horizon; each combination is scored by its mean validation RMSE (original `SWC_20` units) across the folds,
  and the lowest mean wins.
- **Final training:** Adam, MSE loss, up to **20 epochs** with early stopping (**patience 5**) on the 10%
  validation block, one model per horizon, using that horizon's selected features and retuned configuration.
- **Evaluation metrics:** MSE, RMSE, MAE, SMAPE, RSE, CORR, plus MAPE for reference. Targets that are not
  `observed` `SWC_20` readings are excluded from training as well as from validation and test scoring (and
  counted).
- **Forward feature selection:** greedy, separately for 48h and 72h, scored by the same `N_FOLDS`-fold CV
  mean validation RMSE. It starts from `SWC_20` + `Ppt_log` (never removed), adds the remaining candidate or
  feature group that lowers CV RMSE the most, and stops as soon as the best addition no longer improves it.
  The test set is never used for selection.

In [ ]:
# --- Setup and reproducibility ---
import json
import os
import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

PARQUET_PATH = "TxSON33_modeling_complete.parquet"
STATION = "FD11"
START_DATE = "2018-01-01"
END_DATE_EXCLUSIVE = "2021-01-01"      # three calendar years, 2018-2020 inclusive
PERIOD_LABEL = f"{START_DATE[:4]}-{int(END_DATE_EXCLUSIVE[:4]) - 1}"   # "2018-2020", for titles and file names
TARGET_COL = "SWC_20"
INITIAL_FEATURES = ["SWC_20", "Ppt_log"]   # mandatory inputs: initial tuning (Section 10), never removed by selection
SEASON_COLS = ["season_Winter", "season_Spring", "season_Summer", "season_Fall"]
# Optional candidates for forward feature selection (Section 11). Each entry is added or rejected as a whole, so
# the one-hot season and each sine/cosine pair stay complete. Raw Ppt is not a candidate (Ppt_log carries it).
CANDIDATE_GROUPS = {
    "Ppt_rolling_3h_log": ["Ppt_rolling_3h_log"], "Ppt_rolling_24h_log": ["Ppt_rolling_24h_log"],
    "Tair": ["Tair"], "RH": ["RH"], "Srad": ["Srad"],
    "SWC_5": ["SWC_5"], "SWC_10": ["SWC_10"], "SWC_50": ["SWC_50"],
    "T_5": ["T_5"], "T_10": ["T_10"], "T_20": ["T_20"], "T_50": ["T_50"],
    "Wx": ["Wx"], "Wy": ["Wy"],
    "season": SEASON_COLS,
    "hour_cyclic": ["hour_sin", "hour_cos"],
    "dayofyear_cyclic": ["dayofyear_sin", "dayofyear_cos"],
}
CANDIDATE_FEATURES = INITIAL_FEATURES + [c for cols in CANDIDATE_GROUPS.values() for c in cols]   # every possible input
WINDOW_SIZE = 24                       # hours of past data used as input
HORIZONS = [48, 72]                    # hours after the window's last hour; one value predicted per horizon
EXPERIMENT_NAMES = {48: "Experiment A", 72: "Experiment B"}   # each horizon is tuned and evaluated independently

EPOCHS = 20            # max epochs for every main training/testing run
PATIENCE = 5           # early stopping: stop after 5 epochs without validation improvement
TUNING_EPOCHS = 20     # max epochs for the cross-validated grid search and feature selection (Sections 10-11)
N_FOLDS = 2  # Allowed values: 2, 3, or 4
if N_FOLDS not in (2, 3, 4):
    raise ValueError(f"N_FOLDS must be 2, 3 or 4 (TimeSeriesSplit folds inside the training data), got {N_FOLDS!r}")
MIN_RMSE_IMPROVEMENT = 1e-6   # numerical tolerance: a feature is kept only if mean CV RMSE drops by more than this

# Hyperparameter grid (8 combinations), searched per horizon on INITIAL_FEATURES and again on the selected features
PARAM_GRID = {
    "num_layers": [1, 2],
    "hidden_size": [32, 48],
    "lr": [0.001, 0.005],
    "batch_size": [16],
}

# Grid-search tables, selection history and selected features are saved here, one file per horizon.
# In Colab, point this at a mounted Drive folder (e.g. "/content/drive/MyDrive/txson_outputs") to keep the files.
OUTPUT_DIR = "outputs_48h_72h_3yr"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def output_path(horizon, name):
    return os.path.join(OUTPUT_DIR, f"{STATION}_{PERIOD_LABEL}_h{horizon}_{name}")

## 1. Load and inspect the data

Loading the parquet file, filtering to one station and the 2018–2020 period (the station's full history is
kept aside for the rolling rainfall features), and checking exactly what that slice contains — including its `*_Origin` composition — before doing anything to it.

In [ ]:
raw = pd.read_parquet(PARQUET_PATH)
raw["Timestamp"] = pd.to_datetime(raw["Timestamp"])

# Full, chronologically sorted station history -- kept so the rolling rainfall features (Section 2) can use the
# hours before START_DATE -- and the modeling period START_DATE <= Timestamp < END_DATE_EXCLUSIVE
station_df = raw[raw["Station"] == STATION].copy()
station_df = station_df.sort_values("Timestamp").set_index("Timestamp")
df = station_df[(station_df.index >= START_DATE) & (station_df.index < END_DATE_EXCLUSIVE)].copy()

RAW_INPUT_COLS = ["SWC_20", "Ppt", "Wind speed", "Wind direction", "Tair", "RH", "Srad",
                  "SWC_5", "SWC_10", "SWC_50", "T_5", "T_10", "T_20", "T_50"]
expected_rows = len(pd.date_range(START_DATE, END_DATE_EXCLUSIVE, freq="h", inclusive="left"))

print("Shape:", df.shape, f"(a complete hourly {PERIOD_LABEL} has {expected_rows} rows)")
print("Date range:", df.index.min(), "to", df.index.max())
print("Duplicate timestamps:", df.index.duplicated().sum())
print("Hourly and gap-free:", (df.index.to_series().diff().dropna() == pd.Timedelta("1h")).all())
print(f"Full {STATION} history (for rolling features):", station_df.index.min(), "to", station_df.index.max())
print()
print("Missing values in the columns used below:")
print(df[RAW_INPUT_COLS].isna().sum())
print()
print("Origin composition for the target and the precipitation feature:")
print(df["SWC_20_Origin"].value_counts())
print(df["Ppt_Origin"].value_counts())
print()
print("Origin composition of every raw input column (modeling_completion = synthetic):")
print(pd.DataFrame({c: df[f"{c}_Origin"].value_counts() for c in RAW_INPUT_COLS}).T)
print()
print("SWC_20 basic stats:")
print(df["SWC_20"].describe())


**Confirms the plan above:** 26,304 hourly rows for 2018–2020 with zero missing values and no duplicate/gap
timestamps. `SWC_20` is `observed` for every hour except 2 `pipeline_imputed` ones (2019-10-24 16:00 and
2019-11-18 12:00, both inside the training block) and `Ppt` for every hour except 1 (2019-10-24 16:00); neither
has any `modeling_completion` values, so evaluation later is against real sensor readings. As a guard, any
training, validation or test window whose target hour is not an `observed` `SWC_20` reading is dropped before
fitting or scoring (so the 2 imputed training-block hours are never used as training targets), and the number
dropped is reported (Sections 5 and 10).

Note the other inputs: at `FD11` the meteorological columns (`Tair`, `RH`, `Srad`, `Wind speed`,
`Wind direction`) are **100% `modeling_completion`** (synthetic) for 2018–2020. They remain feature-selection
candidates — inputs are not ground truth — but a selected meteorological feature carries modeled, not
measured, information.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
axes[0].plot(df.index, df["SWC_20"], linewidth=0.7)
axes[0].set_ylabel("SWC_20")
axes[0].set_title(f"{STATION}, {PERIOD_LABEL}: soil moisture (target) and precipitation")
axes[1].plot(df.index, df["Ppt"], linewidth=0.7, color="tab:orange")
axes[1].set_ylabel("Ppt")
axes[1].set_xlabel("Date")
plt.tight_layout()
plt.show()


`SWC_20` shows a slow-drydown-with-sharp-rises pattern — soil moisture creeps down between rain events and
jumps up when it rains. The drydown depends on how wet the soil has been over the preceding hours, and the rises
line up with precipitation. That is the kind of ordered, history-dependent signal an LSTM's memory (cell state)
is designed to carry, and it is why precipitation is included as an input from the start.

## 2. Preprocessing and feature engineering

Reproducing the relevant ideas from the project's feature-engineering conventions, written out plainly
rather than imported, keeping only what's actually used below.

### Precipitation

`log1p(Ppt)` compresses the long right tail (mostly zero, occasional storms) into a more model-friendly
range. `log1p` is undefined for values below -1, and a "did it rain" flag should never fire on sensor
noise, so negative readings are clipped to zero first — as a defensive step, not because this
particular station and period needs it (checked explicitly below).

### Rolling rainfall

`Ppt_rolling_3h_log` and `Ppt_rolling_24h_log` are `log1p` of the total rainfall over the 3 / 24 hours
**before** each hour (`shift(1)`: the current hour is excluded, since `Ppt_log` already carries it). They are
computed on the full, chronologically sorted `FD11` history and only then attached to the 2018–2020 rows, so
the first hours of 2018 are initialized from real late-2017 rainfall. A missing hour is never counted as zero
rain: each sum needs every hour of its window (`min_periods` = window length, on a complete hourly grid), so
an incomplete window stays `NaN`. The warm-up check below reports such rows and trims them from the start of
the period (none are expected for `FD11` 2018–2020, whose history goes back to 2014).

In [ ]:
n_negative = int((df["Ppt"] < 0).sum())
print(f"Negative Ppt readings found: {n_negative} out of {len(df)} ({n_negative/len(df)*100:.2f}%)")

df["Ppt"] = df["Ppt"].fillna(0.0).clip(lower=0.0)   # defensive; no-op for this station/period
df["Ppt_log"] = np.log1p(df["Ppt"])


def rolling_ppt_log(ppt, hours):
    """log1p of the total rainfall over the `hours` hours BEFORE each timestamp (the current hour is excluded).

    ppt: chronologically sorted hourly series. It is put on a complete hourly grid first, so a missing row or a
    NaN reading makes that window's sum NaN (min_periods = hours) instead of silently counting as zero rain.
    """
    ppt = ppt.asfreq("h")
    return np.log1p(ppt.shift(1).rolling(hours, min_periods=hours).sum())


ROLLING_PPT_COLS = ["Ppt_rolling_3h_log", "Ppt_rolling_24h_log"]
# Computed on the full station history (negatives clipped as above, missing values left as NaN) and then
# assigned to the 2018-2020 rows, aligned on Timestamp
history_ppt = station_df["Ppt"].clip(lower=0.0)
df["Ppt_rolling_3h_log"] = rolling_ppt_log(history_ppt, 3)
df["Ppt_rolling_24h_log"] = rolling_ppt_log(history_ppt, 24)

# Explicit warm-up handling: NaN means part of that hour's rolling window is missing from the history
incomplete = df[ROLLING_PPT_COLS].isna().any(axis=1)
print(f"Rows whose rolling-rainfall window is incomplete: {int(incomplete.sum())} of {len(df)}")
if incomplete.any():
    first_complete = incomplete.idxmin()
    assert not incomplete.loc[first_complete:].any(), "incomplete rolling windows after the warm-up; check Ppt gaps"
    print(f"Dropping {int(incomplete.loc[:first_complete].sum())} leading warm-up rows; "
          f"modeling starts at {first_complete}")
    df = df.loc[first_complete:].copy()

df[["Ppt", "Ppt_log"] + ROLLING_PPT_COLS].describe().round(3)


### Wind

`Wind speed` and `Wind direction` are stored with spaces in their column names in this dataset (not
`Windspeed`/`Winddirection` as in the older CSVs), so bracket access is needed:
`df["Wind speed"]` rather than `df.Windspeed`. Direction is also circular — 359° and 1° are nearly the
same direction, but as raw numbers they're almost as far apart as possible — so speed+direction is
converted into a 2D vector `(Wx, Wy)` via trigonometry. This also handles calm wind correctly: when
`Wind speed` is near 0, both `Wx` and `Wy` shrink toward 0 regardless of direction, since a motionless
reading carries no directional information. `Wx` and `Wy` are optional candidates for forward feature
selection (Section 11).

In [ ]:
wv = df["Wind speed"].fillna(0.0).clip(lower=0.0)
wd_rad = df["Wind direction"] * np.pi / 180.0
df["Wx"] = wv * np.cos(wd_rad)
df["Wy"] = wv * np.sin(wd_rad)

df[["Wind speed", "Wind direction", "Wx", "Wy"]].head(3)


### Calendar features

Derived from `Timestamp` (the index), so they are known in advance for every hour:

- **Meteorological season**, one-hot: Winter = Dec–Feb, Spring = Mar–May, Summer = Jun–Aug, Fall = Sep–Nov.
  Four numeric 0/1 columns — the text label is never passed to the LSTM.
- **Hour of day** as `hour_sin` / `hour_cos` (period 24 h), so 23:00 and 00:00 are neighbours.
- **Day of year** as `dayofyear_sin` / `dayofyear_cos` (period = that year's length, 365 or 366 days), so
  31 Dec and 1 Jan are neighbours.

In forward feature selection each encoding is one group (`season`, `hour_cyclic`, `dayofyear_cyclic`), so it
is always added or rejected as a whole.

In [ ]:
def add_calendar_features(data_df):
    """Adds one-hot meteorological season and hour / day-of-year sine-cosine pairs, from the Timestamp index."""
    idx = data_df.index
    season = np.select([idx.month.isin([12, 1, 2]), idx.month.isin([3, 4, 5]), idx.month.isin([6, 7, 8])],
                       ["Winter", "Spring", "Summer"], default="Fall")
    for col in SEASON_COLS:                                   # season_Winter, season_Spring, ...
        data_df[col] = (season == col.removeprefix("season_")).astype(np.float32)
    data_df["hour_sin"] = np.sin(2 * np.pi * idx.hour / 24)
    data_df["hour_cos"] = np.cos(2 * np.pi * idx.hour / 24)
    days_in_year = np.where(idx.is_leap_year, 366, 365)
    data_df["dayofyear_sin"] = np.sin(2 * np.pi * (idx.dayofyear - 1) / days_in_year)
    data_df["dayofyear_cos"] = np.cos(2 * np.pi * (idx.dayofyear - 1) / days_in_year)
    return data_df


CYCLIC_COLS = ["hour_sin", "hour_cos", "dayofyear_sin", "dayofyear_cos"]
df = add_calendar_features(df)
assert (df[SEASON_COLS].sum(axis=1) == 1).all()   # exactly one season per hour

print("Hours per season:", df[SEASON_COLS].sum().astype(int).to_dict())
df[CYCLIC_COLS].iloc[[0, 6, 12, 18]].round(3)     # the first day at 00:00, 06:00, 12:00 and 18:00

### Missing data

Confirmed above: zero missing values in 2018–2020, and `SWC_20`/`Ppt` are `observed` except for 2 and 1
`pipeline_imputed` hours, so no imputation logic is needed here — worth stating explicitly rather than
silently assuming it always holds, since large parts of this dataset (per the README) do rely on
`pipeline_imputed` or `modeling_completion` values. The only `NaN`s this notebook can create are incomplete
rolling-rainfall windows, handled explicitly in the precipitation step; the scaler cell below also asserts
that no model input is `NaN`.

## 3. Chronological train / validation / test split (70 / 10 / 20)

Chronological, not random: training data must come entirely before validation data, which must come entirely
before test data, so the model is never evaluated on something that (even indirectly) informed its training.

- **Train (70%)** — used to fit the model. The grid search (Section 10), greedy feature selection and
  retuning (Section 11) all run on cross-validation folds *inside* this block only.
- **Validation (10%)** — used only for early stopping of the final models (picks the best epoch).
- **Test (20%)** — touched once, at the very end, to compare the BiLSTM against the naive baselines.

In [ ]:
n = len(df)
n_train = int(n * 0.70)
n_val = int(n * 0.10)

train_df = df.iloc[:n_train]
val_df = df.iloc[n_train:n_train + n_val]
test_df = df.iloc[n_train + n_val:]

print(f"train: {len(train_df):5d} rows  {train_df.index.min()} -> {train_df.index.max()}")
print(f"val:   {len(val_df):5d} rows  {val_df.index.min()} -> {val_df.index.max()}")
print(f"test:  {len(test_df):5d} rows  {test_df.index.min()} -> {test_df.index.max()}")
assert train_df.index.max() < val_df.index.min() < val_df.index.max()
assert val_df.index.max() < test_df.index.min()

print()
print("Test-period SWC_20 origin composition (must be observed ground truth, not synthetic):")
print(test_df["SWC_20_Origin"].value_counts())

## 4. Normalization

Using `MinMaxScaler`, fit on the training split only, then reused (never refit) to transform
validation and test data.

Fitting the scaler on validation or test data would leak information about their distribution into
preprocessing the model is supposed to be blind to, and would make validation/test performance look
artificially well-behaved. Fitting on train-only means validation/test values can legitimately fall
outside `[0, 1]` if that period is wetter or drier than training — expected, not a bug.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

assert not df[CANDIDATE_FEATURES].isna().any().any(), "a model input contains NaN"

scaler = MinMaxScaler()
scaler.fit(train_df[CANDIDATE_FEATURES])   # fit on TRAIN ONLY; per-column, so any feature subset is scaled the same

def scale(split_df, cols=CANDIDATE_FEATURES, fitted_scaler=scaler):
    arr = fitted_scaler.transform(split_df[cols]).astype(np.float32)
    return pd.DataFrame(arr, columns=cols, index=split_df.index)

train_scaled = scale(train_df)
val_scaled = scale(val_df)
test_scaled = scale(test_df)

# train should be exactly [0, 1]; val/test may fall outside [0, 1] (proves no refit happened)
pd.DataFrame({"train_min": train_scaled.min(), "train_max": train_scaled.max(),
              "val_min": val_scaled.min(), "val_max": val_scaled.max(),
              "test_min": test_scaled.min(), "test_max": test_scaled.max()}).round(3)

## 5. Building X/Y windows

The core idea: slide a fixed-size 24-hour window across the hourly data; each window position becomes
one training example `X`, and its label `y` is the soil-moisture value some fixed number of hours
(`horizon`) after the window's last hour.

Written as an explicit loop rather than a vectorized one-liner, so every step is visible.

Windows are always built **separately inside each partition** (train / validation / test, and each CV fold's
training / validation part), so an input window and its target never straddle a boundary. Before any training,
validation or test use, windows whose target hour is not an `observed` `SWC_20` reading are dropped
(`observed_target_mask`). The mask is applied to the finished windows, never to the source rows, so the hourly
series stays continuous and every kept window still ends exactly `horizon` hours before its target.

In [ ]:
def make_windows(data_df, feature_cols, target_col, window_size, horizon):
    """
    data_df: chronologically ordered, already-scaled dataframe.
    window_size: number of past hours used as input (X).
    horizon: how many hours AFTER the window's last hour the target sits.

    Returns X (num_samples, window_size, num_features), y (num_samples,), and the row-index
    (within data_df) of each window's last input hour / its target hour, for verification.
    """
    values = data_df[feature_cols].values
    targets = data_df[target_col].values
    n = len(data_df)

    X, y, end_idx, target_idx = [], [], [], []
    for start in range(0, n - window_size - horizon + 1):
        end = start + window_size          # first index AFTER the window (exclusive)
        tgt = end - 1 + horizon            # horizon hours after the window's LAST hour (end - 1)
        X.append(values[start:end])
        y.append(targets[tgt])
        end_idx.append(end - 1)
        target_idx.append(tgt)

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32), end_idx, target_idx


In [ ]:
def check_window_alignment(index, end_idx, target_idx, window_size, horizon):
    """Asserts every input covers `window_size` consecutive hourly steps and every target is exactly `horizon`
    hours after its window's last input hour, all inside the partition that `index` belongs to."""
    index = pd.DatetimeIndex(index)
    end_idx, target_idx = np.asarray(end_idx), np.asarray(target_idx)
    assert len(target_idx) > 0, "partition too short for one window"
    assert (np.diff(index.values) == np.timedelta64(1, "h")).all(), "partition is not consecutive hourly data"
    start_idx = end_idx - (window_size - 1)
    assert start_idx.min() >= 0 and target_idx.max() < len(index)       # nothing outside this partition
    assert ((index[end_idx] - index[start_idx]) == pd.Timedelta(hours=window_size - 1)).all()
    assert ((index[target_idx] - index[end_idx]) == pd.Timedelta(hours=horizon)).all()


def observed_target_mask(split_index, target_idx):
    """True for windows whose target hour has an observed SWC_20 reading. pipeline_imputed and synthetic
    modeling_completion values are never used as validation or test ground truth."""
    origin = df[f"{TARGET_COL}_Origin"].reindex(split_index).to_numpy()
    return origin[np.asarray(target_idx, dtype=int)] == "observed"

In [ ]:
# Worked example with real timestamps, for each forecast horizon
for h in HORIZONS:
    X_demo, y_demo, end_idx_demo, target_idx_demo = make_windows(
        train_scaled, INITIAL_FEATURES, TARGET_COL, WINDOW_SIZE, horizon=h
    )
    i = 0
    print(f"{h}h horizon -- first input window's timestamps:")
    print("  first input hour: ", train_df.index[0])
    print("  last input hour:  ", train_df.index[end_idx_demo[i]])
    print("  target hour:      ", train_df.index[target_idx_demo[i]])
    gap_hours = (train_df.index[target_idx_demo[i]] - train_df.index[end_idx_demo[i]]).total_seconds() / 3600
    print(f"  hours between window end and target: {gap_hours} (should equal the requested horizon, {h})")
    print("  Resulting shapes: X =", X_demo.shape, " y =", y_demo.shape)
    assert gap_hours == h
    assert X_demo.shape[1:] == (WINDOW_SIZE, len(INITIAL_FEATURES)) and y_demo.shape == (len(X_demo),)
print(f"X dims: (num_samples, window_size={WINDOW_SIZE}, num_features={len(INITIAL_FEATURES)});  y dims: (num_samples,)")

# Every window of every partition: 24 consecutive hourly inputs, target exactly h hours after the last input,
# never across a partition boundary. Also counts the windows whose target is not an observed SWC_20 reading
# (dropped from training and from validation/test scoring).
exclusions = {}
for h in HORIZONS:
    for part_name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
        _, _, end_idx, target_idx = make_windows(part, [TARGET_COL], TARGET_COL, WINDOW_SIZE, h)
        check_window_alignment(part.index, end_idx, target_idx, WINDOW_SIZE, h)
        keep = observed_target_mask(part.index, target_idx)
        exclusions[(h, part_name)] = {"windows": len(keep), "observed_targets": int(keep.sum()),
                                      "non_observed_targets": int((~keep).sum())}
print(f"Window alignment checks passed for horizons {HORIZONS} in train, val and test.")
pd.DataFrame(exclusions).T.rename_axis(["horizon_hours", "partition"])

## 6. A simple bidirectional LSTM, built from scratch

The structure follows the GeeksforGeeks PyTorch LSTM tutorial
([Long Short-Term Memory Networks using PyTorch](https://www.geeksforgeeks.org/deep-learning/long-short-term-memory-networks-using-pytorch/)):
an `nn.LSTM` with `batch_first=True` reads the input sequence, and a single `nn.Linear` layer maps the LSTM's
final summary of the sequence to one predicted value. Two changes from that tutorial: the LSTM is
**bidirectional**, and the final summary is taken from `h_n` rather than `out[:, -1, :]` (explained below).

**Structure:**
```
input        (batch, 24 hours, F features)         <- SWC_20, Ppt_log + selected features; batch_first=True
nn.LSTM(input_size=F, hidden_size=H, num_layers=L, batch_first=True, bidirectional=True)
                                          <- H and L chosen per horizon by cross-validated grid search
  forward direction  reads hour 1 -> hour 24  -> final hidden state h_fwd  (batch, H)
  backward direction reads hour 24 -> hour 1  -> final hidden state h_bwd  (batch, H)
concat [h_fwd, h_bwd]                    (batch, 2H)
Linear(2H -> 1)                           <- single predicted SWC_20 value (scaled)
```

### What each piece does

- **LSTM cell (gates + cell state):** at every hour the LSTM updates two vectors: the *cell state* (long-term
  memory) and the *hidden state* (what it outputs at that step). Three learned gates control this: the
  **forget gate** decides how much of the old memory to keep, the **input gate** decides how much of the new
  hour's information to write, and the **output gate** decides how much of the memory to expose as the hidden
  state. For soil moisture this lets the model carry "it rained 10 hours ago" or "the soil has been drying
  steadily" forward through the window, in order — unlike the old CNN, whose average pooling threw away *when*
  something happened inside the window.
- **`input_size = F`:** number of features per hour (2 for the starting `SWC_20`, `Ppt_log`; more if forward
  selection adds features). Equivalent to the tutorial's `input_dim`.
- **`batch_first=True`:** tensors are `(batch, time, features)`, which is what `make_windows` already produces.
  (The CNN needed a permute to `(batch, channels, time)`; the LSTM does not.)
- **Initial hidden/cell states:** the tutorial creates `h0`/`c0` as zero tensors explicitly. PyTorch uses zeros
  automatically when none are passed, so the model below omits them — same behavior, less code. Each window
  starts from a blank memory.
- **`bidirectional=True`:** a second LSTM reads the same 24 hours in reverse. This is **not leakage**: at
  prediction time the whole 24-hour input window has already been observed, and the backward direction only
  reads hours *inside* that window — never the target hour, which is `h` hours after the window ends. The two
  directions give two summaries: one that ends on the most recent hour (the forward pass, usually the most
  informative for forecasting) and one that ends on the oldest hour.
- **Why `h_n` instead of the tutorial's `out[:, -1, :]`:** for a one-directional LSTM those are identical. For a
  bidirectional LSTM they are not: `out[:, -1, :]` holds the forward state after all 24 hours **and the backward
  state after only one hour** (the backward pass *starts* at hour 24). `h_n` holds the final state of each
  direction after it has read all 24 hours, so `h_n[-2]` (forward) and `h_n[-1]` (backward) are the correct
  summaries to concatenate.
- **`num_layers = L`:** with `L > 1` the LSTMs are stacked: the second layer reads the full hour-by-hour output
  sequence of the first layer, which lets it learn patterns built on top of the first layer's patterns. `h_n[-2]`
  and `h_n[-1]` always refer to the *last* layer.
- **`Linear(2H -> 1)`:** like the tutorial's `fc` layer: it combines the 2H-number summary into one regression
  output, with no activation function on the output (it's a regression, not a classification). The tutorial
  returns `(out, hn, cn)`; this model returns only the prediction, so the training loop stays a plain
  `pred = model(xb)`.
- **What the single output represents:** the predicted (scaled) `SWC_20` value at `window_end + horizon`.

### Hyperparameters, and why these starting values

| Setting | Value(s) | Why |
|---|---|---|
| `hidden_size` | grid: 32, 48 per direction | The tutorial uses 100 for a 1-feature sine wave; soil-moisture LSTM papers typically use 64–256, but with many more inputs and much more data. A small range keeps the 8-combination grid affordable; `N_FOLDS`-fold CV picks the value per horizon. |
| `num_layers` | grid: 1, 2 | The tutorial uses 1. A 2-layer stack had the lowest validation loss in the earlier single-split run; stack depth is searched because Kim et al. (2026) also tuned it. |
| `bidirectional` | True | Requested design. LSTM vs. BiLSTM results in the literature are mixed — some soil-moisture studies find a plain LSTM equally good — so the naive baselines below are the reality check. |
| window | 24 hours | Kept from the earlier notebook. Hourly rainfall + soil-moisture LSTM studies at 20 cm have used comparable lookbacks of roughly half a day to a day. |
| optimizer / loss | Adam, MSE | Same as the tutorial. Minimizing MSE also minimizes RMSE, the tuning and selection score. |
| learning rate | grid: 0.001, 0.005 | Adam's default (1e-3), and 0.005 (between that default and the tutorial's 0.01, the earlier fixed value). |
| batch size | 16 (single grid value) | Smaller batches gave slightly lower validation loss in the earlier single-split run. |
| epochs | 20 final / 20 tuning, early stopping patience 5 | The tutorial trains a fixed 100 epochs with no validation set; here a validation block (or CV fold) picks the best epoch and its weights are restored. |

### Related LSTM soil-moisture work (for context)

- Kim, Lee, Chung et al. (2026), *Development of long short-term memory models using rainfall and soil moisture
  to predict soil moisture dynamics*, PLOS ONE — the closest setup to this notebook: hourly in-situ soil
  moisture (20/40/60 cm) plus hourly rainfall as the only inputs, MinMax scaling, chronological
  train/val/test, grid search over window size, `num_layers`, `hidden_size`, batch size and learning rate.
  Errors grew with forecast horizon, and long horizons preferred *shorter* windows (more history acted like
  noise). [link](https://journals.plos.org/plosone/article?id=10.1371%2Fjournal.pone.0353268)
- *Enhanced global soil moisture prediction through a sampling-weighted sensitive learning strategy applied to
  various LSTM-based models* (Computers & Geosciences, 2025) — compares LSTM, BiLSTM and GRU; BiLSTM benefited
  the most from their weighting strategy.
  [link](https://www.sciencedirect.com/science/article/abs/pii/S0098300425002183)
- Frontiers in Soil Science (2025) — LSTM and BiLSTM for daily soil moisture/ET; both reached R² 0.96–0.98,
  with the plain LSTM slightly ahead. [link](https://www.frontiersin.org/journals/soil-science/articles/10.3389/fsoil.2025.1612908/full)
- *Hybrid LSTM Method for Multistep Soil Moisture Prediction Using Historical Soil Moisture and Weather Data*
  (Preprints, 2025) — multistep LSTM forecasting from past soil moisture and weather.
  [link](https://www.preprints.org/manuscript/202506.2180)

In [ ]:
class SimpleBiLSTM(nn.Module):
    def __init__(self, num_features, hidden_size=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=num_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
        )
        self.fc = nn.Linear(2 * hidden_size, 1)           # 2x: forward + backward summaries concatenated

    def forward(self, x):
        # x: (batch, 24, num_features)
        _, (h_n, _) = self.lstm(x)          # h_n: (num_layers * 2, batch, hidden_size); h0/c0 default to zeros
        h_forward = h_n[-2]                 # last layer, forward direction  (read hour 1 -> 24)
        h_backward = h_n[-1]                # last layer, backward direction (read hour 24 -> 1)
        h = torch.cat([h_forward, h_backward], dim=1)   # -> (batch, 2 * hidden_size)
        return self.fc(h)                   # -> (batch, 1)


_probe_config = {"hidden_size": 48, "num_layers": 2}   # one of the grid architectures, for the checks below
_probe = SimpleBiLSTM(num_features=len(INITIAL_FEATURES), **_probe_config)
n_params = sum(p.numel() for p in _probe.parameters() if p.requires_grad)
print(f"Trainable parameters (F={len(INITIAL_FEATURES)}, {_probe_config}): {n_params}")

# Shape check, and confirm h_n[-2] is the forward state at the LAST hour while h_n[-1] is the backward
# state at the FIRST hour -- i.e. both directions have read the full window.
_x = torch.randn(4, WINDOW_SIZE, len(INITIAL_FEATURES))
_out, (_h_n, _) = _probe.lstm(_x)
H = _probe.lstm.hidden_size
assert _probe(_x).shape == (4, 1)
assert torch.allclose(_h_n[-2], _out[:, -1, :H])   # forward final state == forward output at hour 24
assert torch.allclose(_h_n[-1], _out[:, 0, H:])    # backward final state == backward output at hour 1
print("Output shape (4, 1) and h_n direction checks passed.")

## 7. Evaluation metrics, verified independently

Reproducing the same 5-metric set the project's evaluator reports (MSE, MAE, SMAPE, RSE, CORR), written
from the mathematical definitions rather than copied, and sanity-checked against known values before
trusting them on real data. MAPE is included too, with an explicit caveat.
RMSE (√MSE, in `SWC_20` units) is reported as well — it is the score used for the grid search and feature
selection.

MAPE divides by the actual value, so it blows up when the target is near zero. `SWC_20` here ranges
roughly 0.18-0.5 (never near zero), so MAPE is less dangerous for this particular target than it would
be for something like precipitation — but it's still the least robust of these metrics in general, so
SMAPE is reported as the primary percentage-error metric, with MAPE kept alongside only for reference.

In [ ]:
def compute_metrics(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=np.float64).flatten()
    y_pred = np.asarray(y_pred, dtype=np.float64).flatten()

    mse = np.mean((y_true - y_pred) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_true - y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + eps))) * 100
    smape = np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + eps)) * 100
    rse = np.sum((y_pred - y_true) ** 2) / (np.sum((y_true - y_true.mean()) ** 2) + eps)
    corr = np.corrcoef(y_true, y_pred)[0, 1] if np.std(y_pred) > 1e-12 else float("nan")

    return {"MSE": mse, "RMSE": rmse, "MAE": mae, "MAPE": mape, "SMAPE": smape, "RSE": rse, "CORR": corr}


# Sanity checks against known cases
perfect = compute_metrics([1, 2, 3], [1, 2, 3])
assert perfect["MSE"] == 0 and perfect["MAE"] == 0 and abs(perfect["CORR"] - 1.0) < 1e-9
mape_check = compute_metrics([1.0], [0.0])
assert abs(mape_check["MAPE"] - 100.0) < 1e-6, mape_check["MAPE"]
assert abs(compute_metrics([0.0, 0.0], [3.0, 4.0])["RMSE"] - np.sqrt(12.5)) < 1e-12
print("Metric sanity checks passed: perfect prediction -> zero error, CORR=1; true=1,pred=0 -> MAPE=100%; "
      "RMSE = sqrt(MSE).")


## 8. Training loop

A plain, manual PyTorch loop: no trainer framework, no callback system — just the steps, visible.

In [ ]:
def train_model(model, train_loader, val_loader, epochs=EPOCHS, patience=PATIENCE, lr=1e-3, verbose=False):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    train_losses, val_losses = [], []
    best_val_loss = float("inf")
    best_state = None
    best_epoch = 0
    epochs_without_improvement = 0

    for epoch in range(epochs):
        model.train()                          # training mode
        running_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
        train_loss = running_loss / len(train_loader.dataset)
        train_losses.append(train_loss)

        model.eval()                           # evaluation mode
        val_running = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                pred = model(xb)
                val_running += criterion(pred, yb).item() * xb.size(0)
        val_loss = val_running / len(val_loader.dataset)
        val_losses.append(val_loss)

        if verbose:
            print(f"  epoch {epoch+1:2d}: train_loss={train_loss:.6f} val_loss={val_loss:.6f}")

        if val_loss < best_val_loss - 1e-7:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            best_epoch = epoch + 1
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)      # restore the best-validation epoch

    return model, train_losses, val_losses, best_epoch

In [ ]:
def run_experiment(horizon, feature_cols=INITIAL_FEATURES, build_model=None, splits=None, scaler_obj=scaler,
                   batch_size=32, lr=1e-3, epochs=EPOCHS, patience=PATIENCE, seed=SEED, verbose=False):
    """
    Trains one model for one horizon and evaluates it on the test split.

    feature_cols: input columns (must be columns of the scaled splits and of scaler_obj).
    build_model:  function num_features -> nn.Module; defaults to SimpleBiLSTM with its default settings.
    splits:       (train_scaled, val_scaled, test_scaled); defaults to the 70/10/20 splits built above.
    Training, validation (early stopping) and test windows whose target is not an observed SWC_20 reading are
    dropped.
    """
    t0 = time.time()
    if build_model is None:
        build_model = lambda nf: SimpleBiLSTM(nf)
    train_s, val_s, test_s = splits if splits is not None else (train_scaled, val_scaled, test_scaled)

    Xtr, ytr, _, tr_tgt = make_windows(train_s, feature_cols, TARGET_COL, WINDOW_SIZE, horizon)
    tr_keep = observed_target_mask(train_s.index, tr_tgt)              # never train on non-observed SWC_20
    Xtr, ytr = Xtr[tr_keep], ytr[tr_keep]
    Xv, yv, _, v_tgt = make_windows(val_s, feature_cols, TARGET_COL, WINDOW_SIZE, horizon)
    Xte, yte, _, te_tgt = make_windows(test_s, feature_cols, TARGET_COL, WINDOW_SIZE, horizon)
    v_keep, te_keep = observed_target_mask(val_s.index, v_tgt), observed_target_mask(test_s.index, te_tgt)
    Xv, yv, Xte, yte = Xv[v_keep], yv[v_keep], Xte[te_keep], yte[te_keep]

    g = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(TensorDataset(torch.tensor(Xtr), torch.tensor(ytr).unsqueeze(1)),
                               batch_size=batch_size, shuffle=True, generator=g)
    val_loader = DataLoader(TensorDataset(torch.tensor(Xv), torch.tensor(yv).unsqueeze(1)), batch_size=batch_size)
    test_loader = DataLoader(TensorDataset(torch.tensor(Xte), torch.tensor(yte).unsqueeze(1)), batch_size=batch_size)

    torch.manual_seed(seed)
    model = build_model(len(feature_cols))
    model, train_losses, val_losses, best_epoch = train_model(model, train_loader, val_loader, epochs=epochs,
                                                                patience=patience, lr=lr, verbose=verbose)

    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            preds.append(model(xb.to(DEVICE)).cpu().numpy())
            trues.append(yb.numpy())
    preds = np.concatenate(preds).flatten()
    trues = np.concatenate(trues).flatten()

    # back to the original SWC_20 scale for interpretable metrics
    target_idx = list(scaler_obj.feature_names_in_).index(TARGET_COL)
    col_min, col_max = scaler_obj.data_min_[target_idx], scaler_obj.data_max_[target_idx]
    preds_orig = preds * (col_max - col_min) + col_min
    trues_orig = trues * (col_max - col_min) + col_min

    metrics = compute_metrics(trues_orig, preds_orig)
    metrics["best_val_loss"] = min(val_losses)   # scaled MSE on the validation split -- used for choosing settings
    metrics["runtime_sec"] = time.time() - t0
    metrics["epochs_run"] = len(train_losses)
    metrics["best_epoch"] = best_epoch
    metrics["n_test_targets"] = len(yte)
    metrics["n_excluded_val_test"] = int((~v_keep).sum() + (~te_keep).sum())   # non-observed targets dropped

    return {"metrics": metrics, "y_true": trues_orig, "y_pred": preds_orig,
            "train_losses": train_losses, "val_losses": val_losses}

## 9. Naive baselines: last value and moving average

Two models with nothing to learn, evaluated on **exactly the same test targets** as the BiLSTM (same windows,
same horizons). Soil moisture is very persistent hour to hour, so these are hard to beat at short horizons — a
BiLSTM that doesn't beat them isn't adding anything yet.

- **Last value (persistence):** predict that `SWC_20` at `window_end + h` equals `SWC_20` at the window's last
  hour.
- **Moving average (24h):** predict the mean `SWC_20` over the 24-hour input window. Smoother than last value,
  but lags behind trends.

Computed directly from the **unscaled** test split, so no inverse scaling is involved. Windows whose target hour
is not an `observed` `SWC_20` reading are dropped exactly as for the BiLSTM.

In [ ]:
def naive_baselines(data_df, horizon, ma_window=WINDOW_SIZE):
    """Returns the true targets and naive predictions for every window with an observed target, in original
    SWC_20 units."""
    X, y, _, target_idx = make_windows(data_df, [TARGET_COL], TARGET_COL, WINDOW_SIZE, horizon)
    keep = observed_target_mask(data_df.index, target_idx)   # same targets as the BiLSTM test scoring
    swc = X[keep, :, 0]
    return y[keep], {
        "Last value": swc[:, -1],
        f"Moving average ({ma_window}h)": swc[:, -ma_window:].mean(axis=1),
    }


naive_results = {}   # naive_results[h][name] = {"metrics": ..., "y_pred": ...}
for h in HORIZONS:
    y_true_naive, naive_preds = naive_baselines(test_df, h)
    naive_results[h] = {name: {"metrics": compute_metrics(y_true_naive, pred), "y_pred": pred, "y_true": y_true_naive}
                        for name, pred in naive_preds.items()}

NAIVE_NAMES = list(naive_results[HORIZONS[0]].keys())
naive_table = pd.concat(
    {name: pd.DataFrame({h: naive_results[h][name]["metrics"] for h in HORIZONS}).T for name in NAIVE_NAMES},
    names=["baseline", "horizon_hours"])
naive_table

## 10. Grid search over the BiLSTM hyperparameters with time-series cross-validation (per horizon)

All tuning happens **inside the 70% training block** — the 10% validation and 20% test blocks are never seen
here, so the final test numbers stay an honest estimate.

**How the folds work (`sklearn.model_selection.TimeSeriesSplit`, `N_FOLDS` folds — 2, 3 or 4, default 2):** the
training block is cut into `N_FOLDS + 1` consecutive chunks. Fold 1 trains on chunk 1 and validates on chunk 2;
fold 2 trains on chunks 1–2 and validates on chunk 3; and so on, up to the last fold, which trains on chunks
1–`N_FOLDS` and validates on the final chunk. Each fold always trains on the past and validates on the period
right after it, so no future information leaks into training (plain `KFold` would sometimes train on later months
and validate on earlier ones). The same folds are used for the initial grid search, feature selection and
retuning.

**Inside each fold:**
- a new `MinMaxScaler` is fit on that fold's training part only, then applied to its validation part,
- windows are built separately for each part, so no input window or target crosses the fold boundary,
- training and validation windows whose target is not an `observed` `SWC_20` reading are dropped (the counts
  are printed below),
- the model trains for up to **20 epochs** (`TUNING_EPOCHS`) with patience 5, the best-validation epoch's
  weights are restored, and the fold's score is the **RMSE of its validation predictions converted back to
  original `SWC_20` units**.

**Choosing a configuration:** `ParameterGrid(PARAM_GRID)` enumerates all 8 combinations:

| Setting | Values |
|---|---|
| `num_layers` | 1, 2 |
| `hidden_size` | 32, 48 |
| `lr` | 0.001, 0.005 |
| `batch_size` | 16 |

Every combination is evaluated on the same `N_FOLDS` folds with the fixed starting inputs
`INITIAL_FEATURES = ["SWC_20", "Ppt_log"]`. Its score is the **mean validation RMSE across the folds**
(fold RMSEs, their standard deviation and each fold's best epoch are recorded too); the lowest mean wins.
This is done **separately for 48h (Experiment A) and 72h (Experiment B)** — the horizon is passed explicitly
and nothing is shared between the two. Cost: 8 × `N_FOLDS` model fits per horizon (16 with the default 2
folds, 32 with 4; a GPU runtime helps).

In [ ]:
from sklearn.model_selection import ParameterGrid, TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=N_FOLDS)
cv_folds = list(tscv.split(train_df))
assert len(cv_folds) == N_FOLDS

for k, (tr_idx, va_idx) in enumerate(cv_folds, start=1):
    tr, va = train_df.iloc[tr_idx], train_df.iloc[va_idx]
    assert tr.index.max() < va.index.min()   # always train on the past, validate on the future
    dropped = {}
    for h in HORIZONS:                       # windows are built inside each part: nothing crosses the boundary
        for part_name, part in (("train", tr), ("val", va)):
            _, _, end_idx, target_idx = make_windows(part, [TARGET_COL], TARGET_COL, WINDOW_SIZE, h)
            check_window_alignment(part.index, end_idx, target_idx, WINDOW_SIZE, h)
            dropped[f"{h}h {part_name}"] = int((~observed_target_mask(part.index, target_idx)).sum())
    print(f"fold {k}: train {len(tr):5d} rows {tr.index.min():%Y-%m-%d} -> {tr.index.max():%Y-%m-%d} | "
          f"val {len(va):5d} rows {va.index.min():%Y-%m-%d} -> {va.index.max():%Y-%m-%d} | "
          f"non-observed targets dropped: {dropped}")
print(f"{len(ParameterGrid(PARAM_GRID))} hyperparameter combinations in PARAM_GRID")

In [ ]:
def build_bilstm(config):
    return lambda nf: SimpleBiLSTM(nf, hidden_size=config["hidden_size"], num_layers=config["num_layers"])


def cv_score(config, horizon, feature_cols=INITIAL_FEATURES, epochs=TUNING_EPOCHS):
    """Validation RMSE of `config` on each TimeSeriesSplit fold, in original SWC_20 units, observed targets only.
    Returns (mean RMSE, per-fold RMSEs, per-fold best epochs)."""
    scale_cols = list(dict.fromkeys(list(feature_cols) + [TARGET_COL]))   # inputs + target, no duplicates
    target_pos = scale_cols.index(TARGET_COL)
    assert len(cv_folds) == N_FOLDS, "N_FOLDS changed: re-run the TimeSeriesSplit cell"
    fold_rmses, best_epochs = [], []
    for tr_idx, va_idx in cv_folds:
        fold_train, fold_val = train_df.iloc[tr_idx], train_df.iloc[va_idx]

        fold_scaler = MinMaxScaler().fit(fold_train[scale_cols])       # fit on this fold's training part only
        tr_s = scale(fold_train, scale_cols, fold_scaler)
        va_s = scale(fold_val, scale_cols, fold_scaler)

        Xtr, ytr, _, tr_tgt = make_windows(tr_s, feature_cols, TARGET_COL, WINDOW_SIZE, horizon)
        tr_keep = observed_target_mask(fold_train.index, tr_tgt)       # never train on non-observed SWC_20
        Xtr, ytr = Xtr[tr_keep], ytr[tr_keep]
        Xv, yv, _, va_tgt = make_windows(va_s, feature_cols, TARGET_COL, WINDOW_SIZE, horizon)
        keep = observed_target_mask(fold_val.index, va_tgt)            # never validate on non-observed SWC_20
        Xv, yv = Xv[keep], yv[keep]
        g = torch.Generator().manual_seed(SEED)
        train_loader = DataLoader(TensorDataset(torch.tensor(Xtr), torch.tensor(ytr).unsqueeze(1)),
                                   batch_size=config["batch_size"], shuffle=True, generator=g)
        val_loader = DataLoader(TensorDataset(torch.tensor(Xv), torch.tensor(yv).unsqueeze(1)),
                                 batch_size=config["batch_size"])

        torch.manual_seed(SEED)
        model = build_bilstm(config)(len(feature_cols))
        model, _, _, best_epoch = train_model(model, train_loader, val_loader, epochs=epochs, patience=PATIENCE,
                                              lr=config["lr"])

        # RMSE of the restored best-epoch model, with its predictions converted back to original SWC_20 units
        model.eval()
        with torch.no_grad():
            pred = torch.cat([model(xb.to(DEVICE)).cpu() for xb, _ in val_loader]).numpy().flatten()
        lo, hi = fold_scaler.data_min_[target_pos], fold_scaler.data_max_[target_pos]
        y_true = fold_val[TARGET_COL].to_numpy()[np.asarray(va_tgt)[keep]]   # observed values, unscaled
        fold_rmses.append(float(compute_metrics(y_true, pred * (hi - lo) + lo)["RMSE"]))
        best_epochs.append(best_epoch)
    return float(np.mean(fold_rmses)), fold_rmses, best_epochs


_cv_cache = {}   # avoids re-running an identical (horizon, features, config) CV evaluation

def cached_cv_score(config, horizon, feature_cols):
    key = (horizon, N_FOLDS, tuple(feature_cols), tuple(sorted(config.items())))
    if key not in _cv_cache:
        _cv_cache[key] = cv_score(config, horizon, feature_cols)
    return _cv_cache[key]


def grid_search_cv(horizon, feature_cols, param_grid=PARAM_GRID, score_fn=cached_cv_score):
    """Scores every ParameterGrid combination on the same CV folds for ONE horizon.
    Returns (table sorted by mean validation RMSE, best config)."""
    grid = list(ParameterGrid(param_grid))
    print(f"--- Grid search: {len(grid)} configs, horizon={horizon}h, features={list(feature_cols)}, "
          f"{N_FOLDS} folds, up to {TUNING_EPOCHS} epochs ---")
    rows = []
    for i, config in enumerate(grid, start=1):
        t0 = time.time()
        mean_rmse, fold_rmses, best_epochs = score_fn(config, horizon, list(feature_cols))
        rows.append({**config, "mean_val_RMSE": mean_rmse, "std_val_RMSE": float(np.std(fold_rmses)),
                     **{f"fold{k}_RMSE": r for k, r in enumerate(fold_rmses, start=1)},
                     **{f"fold{k}_best_epoch": e for k, e in enumerate(best_epochs, start=1)}})
        print(f"  [{i:2d}/{len(grid)}] {config}: mean val RMSE={mean_rmse:.6f}  ({time.time() - t0:.0f}s)")
    best_row = min(rows, key=lambda r: r["mean_val_RMSE"])
    best_config = {k: best_row[k] for k in param_grid}
    print(f"  -> best config for {horizon}h: {best_config} (mean val RMSE={best_row['mean_val_RMSE']:.6f})")
    return pd.DataFrame(rows).sort_values("mean_val_RMSE").reset_index(drop=True), best_config

In [ ]:
# Initial grid search on the fixed starting inputs, separately for each horizon
from IPython.display import display

experiments = {h: {"name": EXPERIMENT_NAMES[h], "horizon": h} for h in HORIZONS}   # one result dict per horizon

for h in HORIZONS:
    exp = experiments[h]
    print(f"\n===== {exp['name']}: initial grid search, {h}h horizon =====")
    exp["initial_grid"], exp["initial_config"] = grid_search_cv(h, INITIAL_FEATURES)
    exp["initial_grid"].to_csv(output_path(h, "initial_grid_search.csv"), index=False)
    display(exp["initial_grid"].head(5))

pd.DataFrame({f"{exp['name']} ({h}h)": {**exp["initial_config"],
                                          "mean_val_RMSE": exp["initial_grid"]["mean_val_RMSE"].iloc[0]}
              for h, exp in experiments.items()}).T

## 11. Greedy forward feature selection, then retuning (48h and 72h, independently)

Scored by the same `N_FOLDS`-fold CV (same folds) as Section 10, inside the training block only — **the test set is never touched
here**. For each horizon, with that horizon's best initial configuration from Section 10 (up to 20 epochs,
patience 5):

1. Start from the mandatory `INITIAL_FEATURES = ["SWC_20", "Ppt_log"]` (never removed) and compute its mean
   CV RMSE.
2. Try adding each remaining entry of `CANDIDATE_GROUPS` on the same folds — a single column, or the
   `season` / `hour_cyclic` / `dayofyear_cyclic` encoding as a whole.
3. Take the addition with the lowest mean CV RMSE and **accept it only if it lowers the current mean RMSE by
   more than `MIN_RMSE_IMPROVEMENT`**.
4. Stop as soon as the best addition does not improve, or when no candidates remain.

Raw `Ppt` is not a candidate (it is only the source of `Ppt_log` and the rolling totals). The history table
records, per step, the candidate tested, whether it was accepted, the RMSE before and after, its fold RMSEs,
the selected features and the stopping reason.

In [ ]:
def greedy_forward_selection(horizon, config, candidate_groups=CANDIDATE_GROUPS, score_fn=cached_cv_score,
                             tol=MIN_RMSE_IMPROVEMENT):
    """Forward selection for ONE horizon, scored only by mean CV RMSE on the training folds (never the test
    set). Starts from the mandatory INITIAL_FEATURES and stops at the first step whose best addition does not
    lower the current mean RMSE by more than `tol`, or when no candidates remain."""
    selected = list(INITIAL_FEATURES)                     # mandatory, never removed
    remaining = dict(candidate_groups)
    current, fold_rmses, _ = score_fn(config, horizon, selected)
    step_rows = [{"step": 0, "candidate": "(start)", "accepted": True, "RMSE_before": np.nan,
                  "candidate_mean_RMSE": current, "RMSE_after": current, "std_val_RMSE": float(np.std(fold_rmses)),
                  **{f"fold{k}_RMSE": r for k, r in enumerate(fold_rmses, start=1)},
                  "selected_features": list(selected), "stop_reason": ""}]
    candidate_scores = {}   # candidate_scores[step][candidate] = mean CV RMSE when that candidate is added
    print(f"Start: {selected}, mean val RMSE={current:.6f}")

    step, stop_reason = 0, "no candidates remain"
    while remaining:
        step += 1
        print(f"\n--- Step {step}: adding one of {len(remaining)} candidates to {selected} ---")
        scores = {}
        for name, cols in remaining.items():
            t0 = time.time()
            scores[name] = score_fn(config, horizon, selected + cols)[:2]   # (mean RMSE, fold RMSEs)
            print(f"  + {name:20s}: mean val RMSE={scores[name][0]:.6f}  ({time.time() - t0:.0f}s)")
        best = min(scores, key=lambda name: scores[name][0])
        candidate_scores[step] = {name: s[0] for name, s in scores.items()}
        best_rmse, best_folds = scores[best]
        accepted = best_rmse < current - tol
        if accepted:
            selected = selected + remaining.pop(best)
        else:
            stop_reason = (f"best addition '{best}' did not lower mean CV RMSE "
                           f"({best_rmse:.6f} vs {current:.6f}, tolerance {tol})")
        step_rows.append({"step": step, "candidate": best, "accepted": accepted, "RMSE_before": current,
                          "candidate_mean_RMSE": best_rmse, "RMSE_after": best_rmse if accepted else current,
                          "std_val_RMSE": float(np.std(best_folds)),
                          **{f"fold{k}_RMSE": r for k, r in enumerate(best_folds, start=1)},
                          "selected_features": list(selected), "stop_reason": ""})
        print(f"  -> {'accepted' if accepted else 'rejected'} {best}: {current:.6f} -> {best_rmse:.6f}")
        if not accepted:
            break
        current = best_rmse

    step_rows[-1]["stop_reason"] = stop_reason
    print(f"Stopped: {stop_reason}. Selected features: {selected}")
    return selected, pd.DataFrame(step_rows).set_index("step"), candidate_scores, stop_reason


for h in HORIZONS:
    exp = experiments[h]
    sel_config = exp["initial_config"]          # this horizon's own best initial configuration
    print(f"\n===== {exp['name']}: feature selection, {h}h horizon, config {sel_config}, {N_FOLDS} folds, "
          f"up to {TUNING_EPOCHS} epochs =====")
    (exp["selected_features"], exp["selection_history"], exp["candidate_scores"],
     exp["stop_reason"]) = greedy_forward_selection(h, sel_config)
    exp["selection_history"].to_csv(output_path(h, "feature_selection_history.csv"))
    with open(output_path(h, "selected_features.json"), "w") as f:
        json.dump({"horizon": h, "selected_features": exp["selected_features"], "stop_reason": exp["stop_reason"],
                   "selection_config": sel_config}, f, indent=2)
    display(exp["selection_history"])

In [ ]:
# Mean CV RMSE of every candidate tried at every step (blank = already selected), per horizon
for h in HORIZONS:
    candidate_table = pd.DataFrame(experiments[h]["candidate_scores"]).T.reindex(columns=list(CANDIDATE_GROUPS))
    candidate_table.index.name = f"{h}h step"
    display(candidate_table.style.format("{:.6f}", na_rep="").highlight_min(axis=1, color="lightgreen"))

In [ ]:
for h in HORIZONS:
    hist = experiments[h]["selection_history"]
    acc, rej = hist[hist["accepted"]], hist[~hist["accepted"]]
    plt.figure(figsize=(11, 4.5))
    plt.errorbar(acc.index, acc["candidate_mean_RMSE"], yerr=acc["std_val_RMSE"], marker="o", capsize=3,
                 label=f"CV mean val RMSE ({N_FOLDS} folds, ± std), accepted")
    if len(rej):
        plt.errorbar(rej.index, rej["candidate_mean_RMSE"], yerr=rej["std_val_RMSE"], marker="x", capsize=3,
                     linestyle="none", color="tab:red", label="best addition, rejected (stop)")
    plt.axvline(acc.index.max(), color="gray", linestyle=":",
                label=f"selected: {len(experiments[h]['selected_features'])} features")
    plt.xticks(hist.index, ["0\nstart" if n == 0 else f"{n}\n+{c}" for n, c in zip(hist.index, hist["candidate"])],
               fontsize=8)
    plt.xlabel("Selection step (candidate added at that step)")
    plt.ylabel("RMSE (SWC_20 units)")
    plt.title(f"Greedy forward feature selection, {h}h horizon ({experiments[h]['name']})")
    plt.legend()
    plt.tight_layout()
    plt.show()

### 11.1 Retuning on the selected features

The same 8-combination grid (Section 10) is run again for each horizon using **only that horizon's selected
features**; the lowest mean `N_FOLDS`-fold CV RMSE (same folds) gives the configuration of the final model. The initial and retuned
results are both kept (and saved) for comparison, and nothing is shared between 48h and 72h. If selection added
nothing, the retune is identical to the initial search and is read from the CV cache instead of re-trained.

In [ ]:
for h in HORIZONS:
    exp = experiments[h]
    print(f"\n===== {exp['name']}: retuning on the selected features, {h}h horizon =====")
    exp["retune_grid"], exp["final_config"] = grid_search_cv(h, exp["selected_features"])
    exp["retune_grid"].to_csv(output_path(h, "retune_grid_search.csv"), index=False)
    with open(output_path(h, "tuning_summary.json"), "w") as f:
        json.dump({"experiment": exp["name"], "horizon": h,
                   "initial_features": INITIAL_FEATURES, "initial_config": exp["initial_config"],
                   "initial_cv_RMSE": float(exp["initial_grid"]["mean_val_RMSE"].iloc[0]),
                   "selected_features": exp["selected_features"], "selection_stop_reason": exp["stop_reason"],
                   "final_config": exp["final_config"],
                   "final_cv_RMSE": float(exp["retune_grid"]["mean_val_RMSE"].iloc[0])}, f, indent=2)

# initial vs. retuned choice, per horizon
pd.DataFrame({f"{exp['name']} ({h}h)": {
    "initial_config": exp["initial_config"], "initial_cv_RMSE": exp["initial_grid"]["mean_val_RMSE"].iloc[0],
    "selected_features": exp["selected_features"],
    "final_config": exp["final_config"], "final_cv_RMSE": exp["retune_grid"]["mean_val_RMSE"].iloc[0]}
    for h, exp in experiments.items()}).T

## 12. Final BiLSTM per horizon (its own selected features and retuned configuration)

Each model is trained on the 70% training block (scaler fit there only), early-stopped on the 10% validation
block (up to 20 epochs, patience 5) and evaluated once on the 20% test block.

In [ ]:
final_results = {}   # final_results[h] = run_experiment output for that horizon's own features and config
for h in HORIZONS:
    exp = experiments[h]
    config, features = exp["final_config"], exp["selected_features"]
    print(f"{exp['name']}: training {h}h model, config {config}, features {features} ...")
    r = run_experiment(h, feature_cols=features, build_model=build_bilstm(config),
                       batch_size=config["batch_size"], lr=config["lr"])
    final_results[h] = r
    print(f"  done in {r['metrics']['runtime_sec']:.1f}s, {r['metrics']['epochs_run']} epochs "
          f"(best epoch {r['metrics']['best_epoch']}); non-observed val/test targets excluded: "
          f"{r['metrics']['n_excluded_val_test']}")

    # the BiLSTM and the naive baselines must be scored on the identical test targets
    assert np.allclose(r["y_true"], naive_results[h][NAIVE_NAMES[0]]["y_true"], atol=1e-5)

## 13. Results by horizon

In [ ]:
from IPython.display import display

METRIC_COLS = ["MSE", "RMSE", "MAE", "MAPE", "SMAPE", "RSE", "CORR"]

def show_horizon(h):
    exp = experiments[h]
    table = pd.DataFrame({f"BiLSTM ({exp['name']})": final_results[h]["metrics"],
                          **{name: naive_results[h][name]["metrics"] for name in NAIVE_NAMES}}).T[METRIC_COLS]
    table.index.name = f"{h}h horizon"
    display(table)
    table.to_csv(output_path(h, "test_metrics.csv"))
    print(f"Features: {exp['selected_features']}\nConfig:   {exp['final_config']}")

    _, _, _, target_idx = make_windows(test_df, [TARGET_COL], TARGET_COL, WINDOW_SIZE, h)
    target_idx = np.asarray(target_idx)[observed_target_mask(test_df.index, target_idx)]   # the scored targets
    target_times = test_df.index[target_idx]
    plt.figure(figsize=(11, 4))
    plt.plot(target_times, final_results[h]["y_true"], label="Actual", linewidth=1.2, color="black")
    plt.plot(target_times, final_results[h]["y_pred"], label=f"BiLSTM ({exp['name']})", linewidth=1)
    for name in NAIVE_NAMES:
        plt.plot(target_times, naive_results[h][name]["y_pred"], label=name, linewidth=0.8, alpha=0.7,
                 linestyle="--")
    plt.title(f"{STATION} {PERIOD_LABEL}: SWC_20 actual vs. predicted, {h}h horizon (test period)")
    plt.ylabel("SWC_20")
    plt.xlabel("Target time")
    plt.legend()
    plt.tight_layout()
    plt.show()

### 13.1 Horizon: 48 hours (Experiment A)

In [ ]:
show_horizon(48)

### 13.2 Horizon: 72 hours (Experiment B)

In [ ]:
show_horizon(72)